In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr, spearmanr

: 

In [ ]:
MATRIX_PATH = Path("../data/processed/motif_matrices/promoter_motif_M_woTFTPM0.csv")

TARGET_COL = "log2_TPM_plus1"
ID_COLS = ["promoter_id", "gene_id"]

TEST_SIZE = 0.10
RANDOM_STATE = 42

: 

In [ ]:
df = pd.read_csv(MATRIX_PATH)

print("Matrix shape:", df.shape)
print("Target column exists:", TARGET_COL in df.columns)

df[["promoter_id", "gene_id", TARGET_COL]].head()

In [ ]:
df = df.dropna(subset=[TARGET_COL]).copy()

feature_cols = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET_COL]
]

X = df[feature_cols]
y = df[TARGET_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing X values:", X.isna().sum().sum())
print("Missing y values:", y.isna().sum())

In [ ]:
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    df[ID_COLS],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
models = {
    "Linear regression": LinearRegression(),
    "Ridge regression": RidgeCV(alphas=np.logspace(-3, 3, 20)),
    "Elastic net": ElasticNetCV(
        alphas=np.logspace(-3, 1, 20),
        l1_ratio=[0.1, 0.5, 0.9],
        max_iter=10000,
        random_state=RANDOM_STATE
    )
}

results = []
predictions = {}

for name, model in models.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    predictions[name] = y_pred

    results.append({
        "model": name,
        "pearson": pearsonr(y_test, y_pred)[0],
        "spearman": spearmanr(y_test, y_pred)[0],
        "r2": r2_score(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred))
    })

results_df = pd.DataFrame(results).sort_values("pearson", ascending=False)
results_df

In [1]:
import sys
print(sys.executable)

/home/okadeeb/miniconda3/envs/fastqc_env/bin/python


In [ ]:
import sys
print(sys.executable)